# 📊 Perfilado Exhaustivo de Datos (Data Profiling) y Análisis de Riesgos de Calidad e Integración
**Curso:** ETL (G01) – Segunda Entrega | **Proyecto:** Análisis Dimensional del IRCA (ODS 6) + API Colombia



---

## 1. Carga de Fuentes Crudas (Sin Transformación)
Cargamos los artefactos exactamente como fueron extraídos en la capa `data/raw/`:
1. **Fuente Primaria (CSV INS):** `Calidad_del_Agua_para_Consumo_Humano_en_Colombia_20260905.csv`
2. **Fuente Secundaria (API Colombia):** `departments_raw.json`, `regions_raw.json`, `cities_raw.json`, `indigenous_reservations_raw.json`

In [1]:
import os
import json
import unicodedata
import pandas as pd
import numpy as np

PRIMARY_CSV = '../data/raw/primary/Calidad_del_Agua_para_Consumo_Humano_en_Colombia_20260905.csv'
API_DIR = '../data/raw/api'

df_raw = pd.read_csv(PRIMARY_CSV, dtype=str)
with open(os.path.join(API_DIR, 'departments_raw.json'), encoding='utf-8') as f:
    depts_raw = json.load(f)
with open(os.path.join(API_DIR, 'regions_raw.json'), encoding='utf-8') as f:
    regions_raw = json.load(f)
with open(os.path.join(API_DIR, 'cities_raw.json'), encoding='utf-8') as f:
    cities_raw = json.load(f)
with open(os.path.join(API_DIR, 'indigenous_reservations_raw.json'), encoding='utf-8') as f:
    indig_raw = json.load(f)

df_depts = pd.DataFrame(depts_raw)
df_regions = pd.DataFrame(regions_raw)
df_cities = pd.DataFrame(cities_raw)
df_indig = pd.DataFrame(indig_raw)

print(f'1. CSV Primario INS cargado: {len(df_raw):,} filas totales x {len(df_raw.columns)} columnas.')
print(f'2. API Colombia (/Department): {len(df_depts)} departamentos x {len(df_depts.columns)} atributos.')
print(f'3. API Colombia (/Region): {len(df_regions)} regiones naturales.')
print(f'4. API Colombia (/City/pagedList): {len(df_cities):,} municipios x {len(df_cities.columns)} atributos.')
print(f'5. API Colombia (/IndigenousReservation/pagedList): {len(df_indig):,} resguardos indígenas x {len(df_indig.columns)} atributos.')

1. CSV Primario INS cargado: 19,160 filas totales x 11 columnas.
2. API Colombia (/Department): 33 departamentos x 18 atributos.
3. API Colombia (/Region): 6 regiones naturales.
4. API Colombia (/City/pagedList): 1,123 municipios x 16 atributos.
5. API Colombia (/IndigenousReservation/pagedList): 829 resguardos indígenas x 13 atributos.


## 2. Perfilado de Esquema, Completitud y Nulos Encubiertos (`'ND'` y `'#TODOS'`) en el CSV Primario
En el archivo del INS no existen celdas vacías tradicionales (`NaN = 0`), sino **literales especiales de texto** que representan dos fenómenos distintos:
1. **`#TODOS` (`569` filas = `2.97%`):** Filas de resumen o subtotal departamental mezcladas con las filas de municipios individuales (`18,591` filas = `97.03%`).
2. **`ND` (No Disponible):** Nulos encubiertos cuando un municipio no reportó muestras en la zona urbana (`1,264` filas = `6.80%`) o en la zona rural (`9,538` filas = `51.30%`).

In [2]:
atomic_df = df_raw[df_raw['Municipio'] != '#TODOS'].copy()
schema_rows = []
for col in df_raw.columns:
    raw_nulls = int(df_raw[col].isna().sum())
    raw_nd = int((df_raw[col] == 'ND').sum())
    raw_todos = int((df_raw[col] == '#TODOS').sum())
    atom_nd = int((atomic_df[col] == 'ND').sum())
    atom_nd_pct = round(atom_nd / len(atomic_df) * 100, 2)
    sample_vals = ', '.join(str(x) for x in atomic_df[col].dropna().unique()[:3])
    schema_rows.append({
        'Columna_CSV': col,
        'Tipo_Leido': str(df_raw[col].dtype),
        'Nulos_NaN_Crudo': raw_nulls,
        'Literal_#TODOS_Crudo': raw_todos,
        'Literal_ND_Crudo': raw_nd,
        'Literal_ND_En_Municipios': atom_nd,
        '%_ND_En_Municipios': f'{atom_nd_pct}%',
        'Valores_Unicos_Municipios': int(atomic_df[col].nunique()),
        'Ejemplos_Observados': sample_vals[:45]
    })
df_schema_csv = pd.DataFrame(schema_rows)
df_schema_csv

Columna_CSV,Tipo_Leido,Nulos_NaN_Crudo,Literal_#TODOS_Crudo,Literal_ND_Crudo,Literal_ND_En_Municipios,%_ND_En_Municipios,Valores_Unicos_Municipios,Ejemplos_Observados
DepartamentoCodigo,str,0,0,0,0,0.0%,33,"11, 13, 15"
Departamento,str,0,0,0,0,0.0%,34,"Bogotá, D.C., Bolívar, Boyacá"
MunicipioCodigo,str,0,569,0,0,0.0%,1095,"11001, 13001, 13006"
Municipio,str,0,569,0,0,0.0%,1015,"Bogotá, D.C., Cartagena de Indias, Achí"
Año,str,0,0,0,0,0.0%,18,"2,024, 2,017, 2,018"
IRCA,str,0,0,0,0,0.0%,952,"3.6, 2.8, 22.8"
Nivel de riesgo,str,0,0,0,0,0.0%,7,"Sin riesgo, Riesgo medio, Alto riesgo"
IRCAurbano,str,0,0,1268,1264,6.8%,967,"1.1, 2.9, 22.8"
Nivel de riesgo urbano,str,0,0,1268,1264,6.8%,8,"Sin riesgo, Riesgo medio, Alto riesgo"
IRCArural,str,0,0,9615,9538,51.3%,993,"14.8, 2.4, ND"


## 3. Evidencia Cuantitativa de la Observación del Profesor en R2 (Por qué `COALESCE(..., 0)` era un error grave)
En el negocio del agua potable en Colombia (Resolución 2115 de 2007), **`0.0%` significa "Sin Riesgo" (Agua 100% potable)**. 
A continuación perfilamos exactamente cuántas filas tienen `'ND'` en una sola zona y mostramos ejemplos reales de municipios en 2024 donde usar `COALESCE(irca_rural, 0)` dividía artificialmente el riesgo a la mitad:

In [3]:
both_valid = int(((atomic_df['IRCAurbano'] != 'ND') & (atomic_df['IRCArural'] != 'ND')).sum())
urb_only = int(((atomic_df['IRCAurbano'] != 'ND') & (atomic_df['IRCArural'] == 'ND')).sum())
rur_only = int(((atomic_df['IRCAurbano'] == 'ND') & (atomic_df['IRCArural'] != 'ND')).sum())
both_nd = int(((atomic_df['IRCAurbano'] == 'ND') & (atomic_df['IRCArural'] == 'ND')).sum())

df_nd_pattern = pd.DataFrame([
    {'Patron_Disponibilidad_Zonal': 'Ambas zonas con medición real (Urbano y Rural válidos)', 'Filas_Municipales': both_valid, 'Porcentaje': f'{round(both_valid/len(atomic_df)*100, 2)}%', 'Impacto_COALESCE_a_0': 'Sin sesgo (ambas zonas existen)'},
    {'Patron_Disponibilidad_Zonal': "Solo Urbano válido y Rural = 'ND' (Sin muestra rural)", 'Filas_Municipales': urb_only, 'Porcentaje': f'{round(urb_only/len(atomic_df)*100, 2)}%', 'Impacto_COALESCE_a_0': 'CRÍTICO: Divide el riesgo urbano entre 2 al sumar 0 en rural'},
    {'Patron_Disponibilidad_Zonal': "Solo Rural válido y Urbano = 'ND' (Sin muestra urbana)", 'Filas_Municipales': rur_only, 'Porcentaje': f'{round(rur_only/len(atomic_df)*100, 2)}%', 'Impacto_COALESCE_a_0': 'CRÍTICO: Divide el riesgo rural entre 2 al sumar 0 en urbano'},
    {'Patron_Disponibilidad_Zonal': "Ambas zonas = 'ND' (Pero IRCA general sí existe)", 'Filas_Municipales': both_nd, 'Porcentaje': f'{round(both_nd/len(atomic_df)*100, 2)}%', 'Impacto_COALESCE_a_0': 'EXTREMO: Convierte el riesgo total del municipio en 0.0% (Agua Óptima)'}
])
df_nd_pattern

Patron_Disponibilidad_Zonal,Filas_Municipales,Porcentaje,Impacto_COALESCE_a_0
Ambas zonas con medición real (Urbano y Rural válidos),8706,46.83%,Sin sesgo (ambas zonas existen)
Solo Urbano válido y Rural = 'ND' (Sin muestra rural),8621,46.37%,CRÍTICO: Divide el riesgo urbano entre 2 al sumar 0 en rural
Solo Rural válido y Urbano = 'ND' (Sin muestra urbana),347,1.87%,CRÍTICO: Divide el riesgo rural entre 2 al sumar 0 en urbano
Ambas zonas = 'ND' (Pero IRCA general sí existe),917,4.93%,EXTREMO: Convierte el riesgo total del municipio en 0.0% (Agua Óptima)


In [4]:
# Ejemplos reales de municipios en 2024 donde IRCArural = 'ND' y cómo COALESCE(..., 0) les reducía el riesgo a la mitad
df_2024 = atomic_df[atomic_df['Año'] == '2,024'].copy()
df_2024['IRCA_num'] = pd.to_numeric(df_2024['IRCA'].str.replace(',', '.'), errors='coerce')
df_2024['Urb_num'] = pd.to_numeric(df_2024['IRCAurbano'].str.replace(',', '.'), errors='coerce')
df_2024['Rur_num'] = pd.to_numeric(df_2024['IRCArural'].str.replace(',', '.'), errors='coerce')
df_2024['Calculo_COALESCE_0_Entrega1'] = (df_2024['Urb_num'].fillna(0) + df_2024['Rur_num'].fillna(0)) / 2
df_2024['Diferencia_Sesgo'] = (df_2024['IRCA_num'] - df_2024['Calculo_COALESCE_0_Entrega1']).round(2)

df_examples_r2 = (
    df_2024[df_2024['IRCArural'] == 'ND']
    .sort_values('IRCA_num', ascending=False)[[
        'Departamento', 'Municipio', 'IRCA', 'IRCAurbano', 'IRCArural', 'Calculo_COALESCE_0_Entrega1', 'Diferencia_Sesgo'
    ]]
    .head(10)
)
df_examples_r2

Departamento,Municipio,IRCA_Consolidado_Real,IRCA_Urbano_Crudo,IRCA_Rural_Crudo,Calculo_COALESCE_0_Entrega1,Diferencia_Sesgo
Antioquia,Zaragoza,96.5,96.5,ND,48.25,48.25
Santander,San Benito,93.9,93.9,ND,46.95,46.95
Chocó,Tadó,93.6,93.6,ND,46.80,46.80
Córdoba,Puerto Escondido,79,79,ND,39.50,39.50
Tolima,Suárez,76.1,76.1,ND,38.05,38.05
Bolívar,Hatillo de Loba,74.2,74.2,ND,37.10,37.10
Bolívar,Margarita,74.1,74.1,ND,37.05,37.05
Cesar,Chimichagua,68.1,68.1,ND,34.05,34.05
Bolívar,Altos del Rosario,66.8,66.8,ND,33.40,33.40
Tolima,Roncesvalles,65.8,65.8,ND,32.90,32.90


**Conclusión de la Celda 4:** Mira a **Zaragoza (Antioquia)** y **Tadó (Chocó)** en 2024: su IRCA real es **`96.5%`** y **`93.6%`** (*Inviable Sanitariamente*), pero como no tuvieron muestra rural (`IRCArural = ND`), el `COALESCE(..., 0)` de la Entrega 1 les asignaba **`48.25%`** y **`46.80%`**, ¡quitándoles casi 50 puntos de riesgo! Por eso en la Entrega 2 incluimos `irca_general` (que tiene `0` nulos) y prohibimos imputar ceros.

## 4. Rangos Numéricos, Percentiles, Dominios Categóricos y Unicidad de Claves en el CSV Primario
Verificamos que ningún registro histórico viole los límites físicos `[0.0, 100.0]` y evaluamos cuál combinación de columnas garantiza unicidad del grano sin colisiones por municipios homónimos.

In [5]:
irca_g = pd.to_numeric(atomic_df['IRCA'].str.replace(',', '.'), errors='coerce')
irca_u = pd.to_numeric(atomic_df['IRCAurbano'].str.replace(',', '.'), errors='coerce')
irca_r = pd.to_numeric(atomic_df['IRCArural'].str.replace(',', '.'), errors='coerce')
anio_n = pd.to_numeric(atomic_df['Año'].str.replace(r'[,\.]', '', regex=True), errors='coerce')

stats_rows = []
for name, s in [('Año (Parseado)', anio_n), ('IRCA General (%)', irca_g), ('IRCA Urbano (%)', irca_u), ('IRCA Rural (%)', irca_r)]:
    stats_rows.append({
        'Variable': name,
        'Conteo_Validos': int(s.notna().sum()),
        'Nulos_o_ND': int(s.isna().sum()),
        'Min': round(float(s.min()), 2),
        'P25': round(float(s.quantile(0.25)), 2),
        'Mediana_P50': round(float(s.median()), 2),
        'Media': round(float(s.mean()), 2),
        'P75': round(float(s.quantile(0.75)), 2),
        'P95': round(float(s.quantile(0.95)), 2),
        'Max': round(float(s.max()), 2),
        'Valores_<0_o_>100': int(((s < 0) | (s > 100)).sum()) if 'IRCA' in name else 0
    })
df_stats_num = pd.DataFrame(stats_rows)
df_stats_num

Variable,Conteo_Validos,Nulos_o_ND,Min,P25,Mediana_P50,Media,P75,P95,Max,Valores_<0_o_>100
Año (Parseado),18591,0,2007.0,2011.0,2015.0,2015.45,2020.0,2024.00,2024.0,0
IRCA General (%),18591,0,0.0,3.2,12.9,21.26,34.3,66.10,100.0,0
IRCA Urbano (%),17327,1264,0.0,1.1,6.2,15.24,20.8,64.00,100.0,0
IRCA Rural (%),9053,9538,0.0,8.4,36.6,38.08,64.4,85.24,100.0,0


In [6]:
# Distribución del Dominio Categórico 'Nivel de riesgo' en los 18,591 registros municipales
df_domains_cat = (
    atomic_df.groupby(['Nivel de riesgo'], dropna=False)
    .size()
    .reset_index(name='Conteo_Registros')
    .assign(Porcentaje=lambda x: (x['Conteo_Registros'] / len(atomic_df) * 100).round(2).astype(str) + '%')
    .sort_values('Conteo_Registros', ascending=False)
)
df_domains_cat

Nivel de riesgo,Conteo_Registros,Porcentaje
Sin riesgo,5890,31.68%
Riesgo medio,4420,23.77%
Riesgo alto,4118,22.15%
Riesgo bajo,3488,18.76%
Inviable sanitariamente,354,1.9%
Bajo riesgo,256,1.38%
Alto riesgo,65,0.35%


In [7]:
# Prueba de Unicidad de Claves Candidatas (Por qué es obligatorio incluir MunicipioCodigo DIVIPOLA)
dup_code_year = int(atomic_df.duplicated(subset=['MunicipioCodigo', 'Año']).sum())
dup_depto_mpio_year = int(atomic_df.duplicated(subset=['Departamento', 'Municipio', 'Año']).sum())
dup_mpio_only_year = int(atomic_df.duplicated(subset=['Municipio', 'Año']).sum())

df_key_uniqueness = pd.DataFrame([
    {'Clave_Candidata_Evaluada': '(MunicipioCodigo, Año)', 'Filas_Evaluadas': len(atomic_df), 'Duplicados_Detectados': dup_code_year, 'Es_Clave_Unica_Valida': 'SÍ (100% Única - Grano Oficial)'},
    {'Clave_Candidata_Evaluada': '(Departamento, Municipio, Año)', 'Filas_Evaluadas': len(atomic_df), 'Duplicados_Detectados': dup_depto_mpio_year, 'Es_Clave_Unica_Valida': 'SÍ (Pero sensible a tildes/mayúsculas)'},
    {'Clave_Candidata_Evaluada': '(Municipio, Año) [SIN Departamento ni Código]', 'Filas_Evaluadas': len(atomic_df), 'Duplicados_Detectados': dup_mpio_only_year, 'Es_Clave_Unica_Valida': f'NO ({dup_mpio_only_year} colisiones por municipios homónimos en distintos departamentos)'}
])
df_key_uniqueness

Clave_Candidata_Evaluada,Filas_Evaluadas,Duplicados_Detectados,Es_Clave_Unica_Valida
"(MunicipioCodigo, Año)",18591,0,SÍ (100% Única - Grano Oficial)
"(Departamento, Municipio, Año)",18591,0,SÍ (Pero sensible a tildes/mayúsculas)
"(Municipio, Año) [SIN Departamento ni Código]",18591,1342,NO (1342 colisiones por municipios homónimos en distintos departamentos)


## 5. Perfilado Exhaustivo de la Nueva Fuente: `API Colombia` (Atributo por Atributo)
Aquí se demuestra visualmente por qué en **API Colombia**:
* El endpoint **`/api/v1/Department`** tiene **`0%` de nulos** en `population` y `surface` (`33/33` completos $
ightarrow$ **ÓPTIMO para R6**).
* En cambio, el endpoint **`/api/v1/City/pagedList`** tiene **`1,060` nulos de `1,123` municipios (`94.39%` de nulos)** en `population`, `surface` y `postalCode`.
* Y el endpoint **`/api/v1/IndigenousReservation/pagedList`** tiene **`0%` de nulos** en `cityId`, `city` y `department` (`829/829` completos $
ightarrow$ **ÓPTIMO para R7**), pero presenta un *typo* en el nombre de la llave plana `deparmentId` (escrito sin la `'t'`).

In [8]:
def profile_api_df(df, endpoint_name):
    rows = []
    n = len(df)
    for col in df.columns:
        null_c = int(df[col].isna().sum())
        null_pct = round(null_c / n * 100, 2)
        non_null_s = df[col].dropna()
        sample_t = type(non_null_s.iloc[0]).__name__ if len(non_null_s) > 0 else 'NoneType'
        rows.append({
            'Endpoint_API': endpoint_name,
            'Atributo_JSON': col,
            'Tipo_Dato_Python': sample_t,
            'Total_Registros': n,
            'No_Nulos': n - null_c,
            'Nulos_Null': null_c,
            '%_Nulos': f'{null_pct}%',
            'Calidad_Para_Integracion': 'ÓPTIMA (100%)' if null_c == 0 else ('CRÍTICA - NO USAR' if null_pct > 50 else 'PARCIAL')
        })
    return pd.DataFrame(rows)

df_api_all_attributes = pd.concat([
    profile_api_df(df_depts[['id', 'name', 'population', 'surface', 'municipalities', 'regionId', 'cityCapitalId']], '/api/v1/Department'),
    profile_api_df(df_cities[['id', 'name', 'departmentId', 'population', 'surface', 'postalCode']], '/api/v1/City/pagedList'),
    profile_api_df(df_indig[['id', 'name', 'code', 'procedureType', 'nativeCommunityId', 'deparmentId', 'department', 'cityId', 'city']], '/api/v1/IndigenousReservation/pagedList')
], ignore_index=True)
df_api_all_attributes

Endpoint_API,Atributo_JSON,Tipo_Dato_Python,Total_Registros,No_Nulos,Nulos_Null,%_Nulos,Calidad_Para_Integracion
/api/v1/Department,id,int64,33,33,0,0.0%,ÓPTIMA (100%)
/api/v1/Department,name,str,33,33,0,0.0%,ÓPTIMA (100%)
/api/v1/Department,population,int64,33,33,0,0.0%,ÓPTIMA (100%)
/api/v1/Department,surface,int64,33,33,0,0.0%,ÓPTIMA (100%)
/api/v1/Department,municipalities,int64,33,33,0,0.0%,ÓPTIMA (100%)
/api/v1/Department,regionId,int64,33,33,0,0.0%,ÓPTIMA (100%)
/api/v1/Department,cityCapitalId,int64,33,33,0,0.0%,ÓPTIMA (100%)
/api/v1/City/pagedList,id,int64,1123,1123,0,0.0%,ÓPTIMA (100%)
/api/v1/City/pagedList,name,str,1123,1123,0,0.0%,ÓPTIMA (100%)
/api/v1/City/pagedList,departmentId,int64,1123,1123,0,0.0%,ÓPTIMA (100%)


### 5.1 Evidencia Visual de los Nulos en `/api/v1/City/pagedList` (`94.39%` vienen con `population = NaN`)

In [9]:
# Muestra de los 1,060 municipios en /api/v1/City donde population, surface y postalCode vienen en NULL (NaN)
df_cities[df_cities['population'].isna()][['id', 'name', 'departmentId', 'population', 'surface', 'postalCode']].head(8)

id,name,departmentId,population,surface,postalCode
87,Puerto Nare,2,NaN,NaN,NaN
88,Puerto Triunfo,2,NaN,NaN,NaN
89,Remedios,2,NaN,NaN,NaN
90,Retiro,2,NaN,NaN,NaN
92,Sabanalarga,2,NaN,NaN,NaN
93,Sabaneta,2,NaN,NaN,NaN
94,Salgar,2,NaN,NaN,NaN
95,San Francisco,2,NaN,NaN,NaN


In [10]:
# Muestra de los únicos 63 municipios en /api/v1/City que sí tienen population reportada
df_cities[df_cities['population'].notna()][['id', 'name', 'departmentId', 'population', 'surface', 'postalCode']].head(8)

id,name,departmentId,population,surface,postalCode
91,Rionegro,2,135465.0,196.0,54040
112,Turbo,2,124552.0,3055.0,57860
143,Arauca,3,96814.0,5841.0,810001
144,Barranquilla,4,2326292.0,520.0,80001
154,Sabanalarga,4,100328.0,399.0,57020
178,Magangué,6,128000.0,1568.0,132511
203,El Carmen de Bolívar,6,70131.0,954.0,132050
217,Tunja,7,180568.0,121.0,150001


## 6. Perfilado de Compatibilidad Cruzada entre Fuentes (Riesgos de Integración CSV <-> API)
Evaluamos qué ocurre si cruzamos directamente los nombres de `Departamento` del CSV del INS contra el campo `name` de `/api/v1/Department`:
1. En el CSV existen **34 nombres de departamento** porque el Archipiélago de San Andrés fue digitado de dos formas distintas (`"... Y Santa Catalina"` con Y mayúscula en 2024 y `"... y Santa Catalina"` con minúscula en años previos).
2. Incluso después de quitar tildes y pasar a mayúsculas, **Bogotá** y **San Andrés** tienen nombres distintos entre el INS y API Colombia, lo que exige un diccionario explícito de homologación en el Contrato de Integración.

In [11]:
def norm_str(val):
    if not isinstance(val, str): return ''
    val = ''.join(c for c in unicodedata.normalize('NFD', val.strip().upper()) if unicodedata.category(c) != 'Mn')
    return ' '.join(val.split())

csv_depts_counts = (
    atomic_df.groupby(['DepartamentoCodigo', 'Departamento'])
    .size()
    .reset_index(name='Registros_CSV')
)
csv_depts_counts['Depto_Normalizado'] = csv_depts_counts['Departamento'].apply(norm_str)

api_depts_map = {norm_str(d['name']): d['name'] for d in depts_raw}
csv_depts_counts['Match_Directo_API'] = csv_depts_counts['Depto_Normalizado'].map(
    lambda x: 'SÍ (Exacto)' if x in api_depts_map else 'NO -> Requiere Homologación'
)
# Mostramos los registros que fallarían en un JOIN directo si no aplicáramos homologación:
csv_depts_counts[csv_depts_counts['Match_Directo_API'] != 'SÍ (Exacto)']

DepartamentoCodigo,Departamento,Registros_CSV,Depto_Normalizado,Match_Directo_API
11,"Bogotá, D.C.",18,"BOGOTA, D.C.",NO -> Requiere Homologación
88,"Archipiélago de San Andrés, Providencia Y Santa Catalina",1,"ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA",NO -> Requiere Homologación
88,"Archipiélago de San Andrés, Providencia y Santa Catalina",32,"ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA",NO -> Requiere Homologación
